# Hurricane Recovery Model (HRM)

## Six hour wind replay and historical recovery planning

Track 1 — Hurricane Preparedness and Recovery. Code[Coogs] AI Hackathon, October 23, 2026.

This notebook reproduces an analysis of the organizer's Greater Houston claim and weather extracts. The finished product combines a six-hour wind hindcast replay, a historical damage atlas, and an editable appointment-capacity scenario. The original financial prediction model failed its check and remains excluded from planning use.

**Question:** How did assessed building damage among eligible single-family flood insurance claims differ across counties and storms?

**Scope:** These are recorded insured claims with known damage, not a sample of all homes. We do not estimate flood probability, uninsured losses, or payout.

### Running this notebook

The displayed outputs were executed with the included original data. On your own computer, extract the project ZIP, open the notebook from that folder, and install `requirements.txt` into your notebook environment. In Colab, run the first cell and select the complete project ZIP when prompted. The browser demo in `demo/index.html` requires no Python setup.

Source folder: https://drive.google.com/drive/folders/1doFLc1xXvUDTBvoDg0wmebjGIaaCCy0a

Dataset brief: https://docs.google.com/document/d/1n9FrZHastJOO1jy2f6X_PbYukTuAnrWgbJugyTK3lkw/edit


In [1]:
from pathlib import Path
import sys, zipfile, importlib.util

ROOT = Path.cwd()
if not (ROOT / 'train_model.py').exists():
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('Open this notebook from the extracted HoustonStormReady project folder.')
    print('Select the complete HoustonRecoveryLens.zip project file.')
    uploaded = files.upload()
    dest = Path('/content/recovery_lens_project')
    dest.mkdir(exist_ok=True)
    for name, raw in uploaded.items():
        import io
        with zipfile.ZipFile(io.BytesIO(raw)) as archive:
            for member in archive.infolist():
                target = (dest / member.filename).resolve()
                if not target.is_relative_to(dest.resolve()):
                    raise ValueError('Unsafe archive path')
            archive.extractall(dest)
    roots = list(dest.rglob('train_model.py'))
    if len(roots) != 1:
        raise RuntimeError('Choose the complete project ZIP, not just the dataset archive.')
    ROOT = roots[0].parent
sys.path.insert(0, str(ROOT))
print('Project files found. Data and analysis are ready.')


Project files found. Data and analysis are ready.


## Load and inspect the original records

The four supplied files are retained unchanged. The claims have 18 columns; the weather has hourly observations for five county reference points during four storms. The boundary file contains nine counties, four of which lack matched weather and claim data here.

If imports fail in a new environment, install the packages from `requirements.txt` and restart the notebook kernel before continuing.


In [2]:
import pandas as pd
import numpy as np
import json
import train_model

claims, eligible, weather, hourly_weather, audit = train_model.prepare()
print('Original claim records:', len(claims))
print('Eligible single-family records:', len(eligible))
print('Hourly weather records:', len(hourly_weather))
print('County/storm groups:', len(weather))
print('\nEligible records by storm:')
print(eligible.groupby('storm').agg(claims=('damage','size'), median_damage=('damage','median')).round(2).to_string())


Original claim records: 80547
Eligible single-family records: 60020
Hourly weather records: 3960
County/storm groups: 20

Eligible records by storm:
          claims  median_damage
storm                          
Beryl       1622         8452.0
Harvey     54701        71170.0
Imelda      3528        40336.0
Nicholas     169         4271.0


## Quality checks and joins

County FIPS are standardized as five-character strings. Occupancy codes 1 and 11 are treated as single-family residences. We restrict to the four named storms with weather and exclude missing or negative damage and insurance coverage outside $1–$250,000. Unknown or uncommon flood-zone values enter `OTHER`.

We merge on storm identity and county using a many-to-one validation. The code asserts that no eligible claim is lost or duplicated in the merge and that model input fields are complete. No unique claim or property ID is supplied, so we cannot reliably remove repeat properties across different storms.


In [3]:
print(json.dumps(audit, indent=2))
assert len(eligible) == 60020
assert eligible[['rain_total','gust_max','coverage','county','zone','damage']].notna().all().all()
assert eligible['damage'].ge(0).all()
assert eligible['coverage'].between(1,250000).all()
print('\nChecks passed: join cardinality, input completeness, target validity, and coverage scope.')


{
  "raw_claims": 80547,
  "selected_single_family_claims": 66992,
  "missing_damage_in_selected": 6941,
  "negative_damage_in_selected": 0,
  "out_of_scope_coverage_in_selected": 426,
  "model_claims": 60020,
  "weather_rows": 3960,
  "weather_groups": 20,
  "exact_duplicate_full_claim_rows": 0,
  "notes": [
    "No unique claim ID supplied; distinct properties and repeat claims cannot be verified.",
    "Missing damage is excluded, never replaced with zero.",
    "County FIPS standardized as five-character strings.",
    "Only five counties have claims and weather; four other boundary counties have no model support.",
    "Recorded claims are not the population of homes and do not measure flood probability."
  ]
}

Checks passed: join cardinality, input completeness, target validity, and coverage scope.


## Summarize historical recovery burden

These are assessed building damage amounts, separate from contents damage and net insurance payment. Amounts are nominal dollars and have not been adjusted for inflation. A median is the middle eligible claim amount. The sums are limited to the selected supplied records.

Storm windows have different lengths. Rainfall comes from full-window historical reanalysis at one reference point per county. It is useful context but is neither a county-wide measurement nor information demonstrated to be available before the storm.


In [4]:
historical = eligible.groupby(['storm','county']).agg(
    claims=('damage','size'), median_damage=('damage','median'),
    recorded_damage_total=('damage','sum'), rainfall_in=('rain_total','first'),
    gust_mph=('gust_max','first')).reset_index()
display_table = historical.copy()
small = display_table['claims'] < 20
display_table.loc[small, ['median_damage','recorded_damage_total']] = np.nan
print(display_table.round(2).to_string(index=False))
print('\nDamage summaries with fewer than 20 eligible records are suppressed above and in the demo.')


   storm     county  claims  median_damage  recorded_damage_total  rainfall_in  gust_mph
   Beryl   Brazoria     211         8200.0              3632331.0         6.21      76.7
   Beryl  Fort Bend      75        13753.0              2204033.0         6.61      76.7
   Beryl  Galveston     450         5818.0              6511799.0         5.20      86.8
   Beryl     Harris     866        11009.5             19289060.0         5.75      78.7
   Beryl Montgomery      20         7588.5               355348.0         5.41      67.1
  Harvey   Brazoria    4151        64335.0            289748615.0        17.48      51.0
  Harvey  Fort Bend    2697        85983.0            257959695.0        16.97      51.0
  Harvey  Galveston    6942        65761.5            540022397.0        20.31      61.7
  Harvey     Harris   38920        71702.5           3630819406.0        21.66      57.0
  Harvey Montgomery    1991        84474.0            192123828.0        23.78      53.9
  Imelda   Brazoria  

## Train and select the models

We separate entire storms rather than randomly mixing claims from the same storm across train and test sets:

- Train on Harvey 2017 and Imelda 2019; assign equal total fitting weight to each storm.
- Select using Nicholas 2021.
- Evaluate the selected model on Beryl 2024, which is excluded from training and selection.

Compare the training median with a shallow decision tree and ridge regression. Inputs are rainfall, peak gust, coverage, county, and recorded flood zone. Payout, damage, loss date, storm name, and indoor water depth are excluded from model inputs. Only two independent training storms remain; many claims do not fix this lack of storm diversity.

The next cell reruns the full analysis and writes the metrics, source hashes, model object, aggregate summaries, and rejected prediction table. Results are deterministic for the recorded environment.


In [5]:
train_model.main()


{
  "selected_model": "Linear ridge regression",
  "selection_metric": "Nicholas 2021 mean absolute error",
  "train_storms": [
    "Harvey 2017",
    "Imelda 2019"
  ],
  "validation_storm": "Nicholas 2021",
  "test_storm": "Beryl 2024",
  "train_n": 58229,
  "validation_n": 169,
  "test_n": 1622,
  "baseline_prediction": 68509.0,
  "validation_candidates": [
    {
      "model": "Training median baseline",
      "mae": 58345.18343195266,
      "median_absolute_error": 64238.0,
      "r2": -15.037640512203026
    },
    {
      "model": "Shallow decision tree",
      "mae": 33864.91049514341,
      "median_absolute_error": 36071.72810532202,
      "r2": -4.997588500240277
    },
    {
      "model": "Linear ridge regression",
      "mae": 10559.600239784972,
      "median_absolute_error": 4840.0,
      "r2": -0.4409796481685966
    }
  ],
  "test_baseline": {
    "mae": 51050.07829839704,
    "median_absolute_error": 60366.0,
    "r2": -3.831803490386765
  },
  "test_model": {
    "ma

## Inspect the later-storm failure

Ridge regression had the lowest validation error, but its Beryl predictions were all clipped to zero. The apparently better score against the training-median baseline is therefore not evidence of useful generalization. An always-zero diagnostic has exactly the same Beryl error.

**Decision: reject the candidate model for planning use.** No model prediction is used in the explorer. Its test R² is negative, and its collapsed outputs prevent it from distinguishing severity among Beryl claims. Additional transformations or tuning after seeing these results would require a fresh final test storm to support an independent performance claim.


In [6]:
metrics = json.loads((ROOT/'results'/'metrics.json').read_text())
print('Selected model:', metrics['selected_model'])
print('Beryl model MAE: $' + format(metrics['test_model']['mae'], ',.2f'))
print('Beryl training-median baseline MAE: $' + format(metrics['test_baseline']['mae'], ',.2f'))
print('Beryl always-zero diagnostic MAE: $' + format(metrics['zero_prediction_baseline_mae'], ',.2f'))
print('Fraction of Beryl predictions clipped to zero:', metrics['test_zero_prediction_share'])
print('Beryl R squared:', round(metrics['test_model']['r2'], 3))
print('Decision:', metrics['model_status'])
assert metrics['test_zero_prediction_share'] == 1.0
assert np.isclose(metrics['test_model']['mae'], metrics['zero_prediction_baseline_mae'])


Selected model: Linear ridge regression
Beryl model MAE: $19,724.15
Beryl training-median baseline MAE: $51,050.08
Beryl always-zero diagnostic MAE: $19,724.15
Fraction of Beryl predictions clipped to zero: 1.0
Beryl R squared: -0.626
Decision: REJECTED FOR PLANNING USE


## Historical recovery scope and next steps

Open `demo/index.html` to explore the observed data. The demo suppresses historical damage summaries with fewer than 20 records, distinguishes unsupported counties, includes data scope beside the visuals, and exports a four-storm all-zone comparison for the selected county. It contains aggregate data and boundaries; it does not expose individual claim coordinates.

Next steps for useful prediction include more independent storms, stronger baselines, property and exposure information, forecast-time weather, and a fresh storm reserved for final evaluation. The provided data does not include the outage target file mentioned in the brief, so no outage-duration claims are made.

### Attribution and reproducibility

The organizer's brief attributes the claims to OpenFEMA NFIP, weather to Open-Meteo ERA5, tracks to NOAA IBTrACS, and boundaries to the U.S. Census Bureau. Original extracts were downloaded on October 23, 2026 and their hashes are recorded in `results/data_manifest.json`. Upstream extraction procedures were not independently reproduced. The hurricane track file is inspected for context and is not used as a separate damage label.

This project used AI coding assistance. Its limitations and failed model results are part of the presentation, not concealed. The current analysis is a retrospective data science project, not an operational emergency or household risk forecast.


## Upgrade Six hour peak gust hindcast

The weather records support a clearer short-term target: the maximum gust during the following six hours. Inputs use only the current hour and preceding six hours. A future-perturbation test confirms that changing future gust, pressure, and rainfall leaves the current feature row unchanged.

Train on Harvey and Imelda, select using Nicholas, replay Beryl as a development check, then evaluate three separately downloaded 2026 windows with the frozen model. Beryl was inspected during prior project development and is not an untouched final test. The candidate set and selection rule were fixed before the added weather was downloaded. Window names and dates are derived from the supplied tracks; their local storm attribution was not independently verified.

Additional source: Open-Meteo Historical Weather API, Copernicus ERA5 reanalysis, CC BY 4.0. Request URLs, explicit mph and inch units, response hashes, and raw responses are in data/external_validation. This is a retrospective hindcast and does not demonstrate issue-time operational forecast performance.


In [7]:
import train_wind_model
train_wind_model.main()


{
  "selection": "Gradient boosting",
  "validation": [
    {
      "model": "Persistence",
      "mae_mph": 4.829848484848484
    },
    {
      "model": "Ridge",
      "mae_mph": 5.318234021021461
    },
    {
      "model": "Residual gradient boosting",
      "mae_mph": 4.692924849917466
    },
    {
      "model": "Gradient boosting",
      "mae_mph": 4.647572694966137
    }
  ],
  "evaluations": [
    {
      "storm": "Beryl",
      "n": 900,
      "mae_mph": 5.607152683254575,
      "rmse_mph": 8.556754627639055,
      "persistence_mae_mph": 6.503555555555556,
      "improvement_pct": 13.783273851412615,
      "peak_decile_mae_mph": 18.880377001700886,
      "peak_decile_mean_bias_mph": -18.191858828116914,
      "observed_max_mph": 86.8,
      "prediction_max_mph": 57.40226379519893,
      "role": "Development replay",
      "residual_band_coverage": 0.7844444444444445
    },
    {
      "storm": "Harvey",
      "n": 1380,
      "mae_mph": 2.7026760288530514,
      "rmse_mph": 3

## Wind results and appointment capacity scenario

The selected gradient boosting model lowered Beryl replay mean absolute error from 6.50 to 5.61 mph, about 13.8%. On the three additional 2026 windows combined, average error fell from 3.68 to 3.53 mph, about 3.9%. These modest retrospective gains are reported without a statistical significance claim. County reference points and overlapping target windows are correlated.

Severe conditions remain difficult: Beryl's highest-gust tenth of windows had about 18.9 mph average error and strong underestimation. The demo exposes that weakness, flags current gusts beyond training support, and reveals the recorded reference outcome on demand. The shaded reference band uses Nicholas residuals and offers no coverage guarantee.

The capacity screen distributes whole appointment slots in proportion to user-editable assumed requests, after a county floor when feasible. It conserves available slots, never exceeds assumed demand, and exports capacity, floor, assumptions, allocations, and remaining requests. It does not predict need, infer uninsured demand, or route emergency resources. Using historical claim mix is an explicitly hypothetical scenario option.


In [8]:
wind_metrics = json.loads((ROOT/'results/wind_metrics.json').read_text())
table = pd.DataFrame(wind_metrics['evaluations'])
print(table[['storm','role','n','mae_mph','persistence_mae_mph','improvement_pct']].round(3).to_string(index=False))
print('\nSevere Beryl-window average error:', round(table.loc[table.storm=='Beryl','peak_decile_mae_mph'].iloc[0],2), 'mph')
assert wind_metrics['selected_model']=='Gradient boosting'
assert np.isfinite(table['mae_mph']).all() and table['mae_mph'].ge(0).all()
print('All additional weather windows are reported; the model is a research replay, not an operational warning system.')


              storm                          role    n  mae_mph  persistence_mae_mph  improvement_pct
              Beryl            Development replay  900    5.607                6.504           13.783
             Harvey                      Training 1380    2.703                4.575           40.920
           Nicholas                    Validation  660    4.648                4.830            3.774
             Imelda                      Training  780    3.164                5.059           37.467
 Arthur 2026 window Additional frozen-model check 1140    4.201                4.616            8.979
 Bertha 2026 window Additional frozen-model check 1620    3.113                3.125            0.372
Edouard 2026 window Additional frozen-model check 1500    3.477                3.559            2.299

Severe Beryl-window average error: 18.88 mph
All additional weather windows are reported; the model is a research replay, not an operational warning system.
